In [1]:
cd /resnick/groups/astuart/bhchen/Learn_DA_selfuse

/resnick/groups/astuart/bhchen/Learn_DA_selfuse


/home/bhchen/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
experiment_configs = [
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_es_joint_CorrTerms_arctan",
        "tags": ["es", "CorrTerms", "arctan"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_es_joint_CorrTerms_square_root",
        "tags": ["es", "CorrTerms", "square_root"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_es_joint_EtE-LRes_arctan",
        "tags": ["es", "EtE-LRes", "arctan"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_es_joint_EtE-LRes_square_root",
        "tags": ["es", "EtE-LRes", "square_root"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_nl2_joint_CorrTerms_arctan",
        "tags": ["nl2", "CorrTerms", "arctan"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_nl2_joint_CorrTerms_square_root",
        "tags": ["nl2", "CorrTerms", "square_root"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_nl2_joint_EtE-LRes_arctan",
        "tags": ["nl2", "EtE-LRes", "arctan"]
    },
    {
        "path": "2026-02-08_15-12lorenz96_1.0_10_60_8192_nl2_joint_EtE-LRes_square_root",
        "tags": ["nl2", "EtE-LRes", "square_root"]
    }
]

In [3]:
import torch
import os
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# =================== Configuration & Style ===================

MAIN_LINE_WIDTH = 2.0
GRID_LS = "--"
GRID_ALPHA = 0.6
OUT_DIR = os.path.join("save", "figures")
os.makedirs(OUT_DIR, exist_ok=True)

# NOTE: Metrics specific to Lorenz 96
metrics_config = [
    {"key": 'mean_rrmse', "ylabel": "RRMSE", "fname": "rrmse"},
    {"key": 'mean_rcrps', "ylabel": "RCRPS", "fname": "rcrps"},
]

def get_method_style(tags):
    """
    Determine color and linestyle based on method tags.
    - Color: Differentiates Base Method + Corr/EtE logic
    - Linestyle: Differentiates arctan vs square_root
    """
    # 1. Linestyle based on the last tag (function type)
    if "arctan" in tags:
        ls = "-"
    elif "square_root" in tags:
        ls = "--"
    else:
        ls = ":"

    # 2. Color logic to distinguish CorrTerms vs EtE-LRes across different bases
    # Matrix: [Base] x [Mechanism]
    if "es" in tags:
        color = "tab:blue" if "CorrTerms" in tags else "tab:cyan"
    elif "nl2" in tags:
        color = "tab:red" if "CorrTerms" in tags else "tab:orange"
    elif "NLL" in tags:
        color = "tab:green" if "CorrTerms" in tags else "tab:olive"
    else:
        color = "black"

    return {"color": color, "linestyle": ls}

# =================== UI Components ===================

all_tags = sorted(list(set(t for c in experiment_configs for t in c["tags"])))

tag_checks = [
    widgets.Checkbox(
        value=True, 
        description=tag, 
        layout=widgets.Layout(width='auto', margin='0px 10px 0px 0px') 
    ) for tag in all_tags
]

# Range Sliders
x_range_slider = widgets.IntRangeSlider(
    value=[0, 500], min=0, max=5000, step=50, 
    description='X (Epoch):', continuous_update=False
)
rrmse_y_slider = widgets.FloatRangeSlider(
    value=[0, 1.0], min=0, max=5.0, step=0.05, 
    description='RRMSE Range:', continuous_update=False
)
rcrps_y_slider = widgets.FloatRangeSlider(
    value=[0, 1.0], min=0, max=5.0, step=0.05, 
    description='RCRPS Range:', continuous_update=False
)

save_check = widgets.Checkbox(value=False, description="Save PDF")
side_legend_check = widgets.Checkbox(value=True, description="Side Legend")
btn_plot = widgets.Button(description="Generate Plots", button_style='primary', icon='area-chart')
output = widgets.Output()

def run_plotting(_):
    with output:
        clear_output(wait=True)
        selected_tags = set([chk.description for chk in tag_checks if chk.value])
        
        if not selected_tags:
            print("Please select tags.")
            return

        active_configs = [
            c for c in experiment_configs 
            if set(c["tags"]).issubset(selected_tags)
        ]
        
        handles, labels = [], []

        for metric in metrics_config:
            fig, ax = plt.subplots(figsize=(12, 5))
            lines_plotted = False 
            
            for config in active_configs:
                folder_path = os.path.join('save', config["path"])
                file_path = os.path.join(folder_path, 'training_records.pt')
                if not os.path.exists(file_path): continue

                data = torch.load(file_path, weights_only=True, map_location="cpu")
                curve = np.array(data.get(metric['key'], []))
                epochs = np.array(data.get('test_epochs', []))
                
                if len(curve) > 0:
                    name = "+".join(config["tags"])
                    style = get_method_style(config["tags"])
                    ln, = ax.plot(epochs, curve, label=name, lw=MAIN_LINE_WIDTH, **style)
                    lines_plotted = True
                    if metric == metrics_config[0]:
                        handles.append(ln)
                        labels.append(name)

            # Axis limits
            ax.set_xlim(x_range_slider.value)
            
            # Use specific slider based on metric
            if metric['key'] == 'mean_rrmse':
                ax.set_ylim(rrmse_y_slider.value)
            else:
                ax.set_ylim(rcrps_y_slider.value)

            ax.set_title(f"Lorenz96 - {metric['ylabel']}", fontsize=15)
            ax.set_ylabel(metric['ylabel'], fontsize=12)
            ax.set_xlabel('Epoch', fontsize=12)
            ax.grid(True, ls=GRID_LS, alpha=GRID_ALPHA)
            
            if side_legend_check.value and lines_plotted:
                ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)

            if save_check.value:
                plt.savefig(os.path.join(OUT_DIR, f"Lorenz96_{metric['fname']}.pdf"), bbox_inches="tight")
            plt.show()

# =================== Layout ===================

tag_rows = [widgets.HBox(tag_checks[i:i + 6]) for i in range(0, len(tag_checks), 6)]
ui = widgets.VBox([
    widgets.HTML("<b>Method Tags</b>"),
    *tag_rows,
    widgets.HTML("<hr><b>Axis Ranges</b>"),
    x_range_slider, rrmse_y_slider, rcrps_y_slider,
    widgets.HTML("<hr>"),
    widgets.HBox([save_check, side_legend_check, btn_plot]),
    output
])

display(ui)
btn_plot.on_click(run_plotting)